# 13 — Evaluación de los modelos exportados (Keras, float32 e int8)

**Objetivo.** Cuantificar la pérdida de exactitud introducida por la exportación a TensorFlow Lite y por la
cuantización a enteros de 8 bits, comparando las tres variantes de cada clasificador sobre el mismo conjunto de
prueba independiente.

**Tecnología.** TensorFlow/Keras y el intérprete de TensorFlow Lite para la inferencia; scikit-learn para las
métricas. No se realiza entrenamiento: se evalúan los modelos ya exportados.

**Metodología.** Para cada imagen del conjunto de prueba se construye una sola vez la doble entrada del sistema
(imagen normalizada con Shades-of-Gray e imagen de hoja aislada mediante la máscara de M_seg) y se presenta sin
modificación a las tres variantes. Los modelos EfficientNet reciben valores en el rango [0, 255], ya que la
normalización está incluida en la propia red. En la variante entera, las entradas y salidas se convierten mediante
los parámetros de cuantización declarados por el intérprete. Se reportan exactitud y F1 macro por variante y su
diferencia respecto del modelo Keras de referencia.

In [ ]:
!pip install -q tensorflow scikit-learn pandas

In [ ]:
from pathlib import Path
import glob
from google.colab import drive

drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/glycine_vision_baselines')
OUT.mkdir(parents=True, exist_ok=True)
DRIVE = Path('/content/drive/MyDrive')


def _first(candidatos, patron=None):
    for c in candidatos:
        if Path(c).exists():
            return Path(c)
    if patron:
        hits = sorted(glob.glob(patron, recursive=True))
        if hits:
            return Path(hits[0])
    return None


SPLIT = _first(['/content/splits'], str(DRIVE / '**' / 'splits'))
assert SPLIT is not None, 'No se encontro la carpeta splits.'
TEST_BIN = _first([SPLIT / 'test' / 'clasificacion_binaria'])
TEST_PAT = _first([SPLIT / 'test' / 'clasificacion_patogeno'])
SEG_PATH = _first([OUT / 'model_seg.keras'], str(DRIVE / '**' / 'model_seg.keras'))

MODELOS = {}
for clave, nombre in [('m1_keras', 'model1_binary.keras'), ('m1_f32', 'model1.tflite'), ('m1_int8', 'model1_int8.tflite'),
                      ('m2_keras', 'model2_pathogen.keras'), ('m2_f32', 'model2.tflite'), ('m2_int8', 'model2_int8.tflite')]:
    MODELOS[clave] = _first([OUT / nombre], str(DRIVE / '**' / nombre))

for clave, ruta in list(MODELOS.items()) + [('segmentador', SEG_PATH), ('test binario', TEST_BIN), ('test patogeno', TEST_PAT)]:
    assert ruta is not None, f'No encontrado: {clave}'
    print(f'{clave:12s} {ruta}')

In [ ]:
import json
import numpy as np
import cv2
import tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

_MSEG = tf.keras.models.load_model(SEG_PATH, compile=False)


def chromatic_normalize(img_rgb):
    x = img_rgb.astype(np.float32)
    il = np.power(np.mean(np.power(x, 6), axis=(0, 1)), 1.0 / 6.0)
    return np.clip(x * np.clip(il.mean() / (il + 1e-6), 0.6, 1.6), 0, 255).astype(np.uint8)


def mseg_mask(img_rgb, size):
    small = chromatic_normalize(cv2.resize(img_rgb, (256, 256)))
    prob = _MSEG.predict((small.astype(np.float32) / 255.0)[np.newaxis], verbose=0)[0]
    leaf = (np.argmax(prob, -1) == 1).astype(np.uint8)
    return cv2.resize(leaf, size, interpolation=cv2.INTER_NEAREST)


def construir_entradas(directorio, size):
    clases = sorted(p.name for p in Path(directorio).iterdir() if p.is_dir())
    idx = {c: i for i, c in enumerate(clases)}
    originales, hojas, etiquetas = [], [], []
    for c in clases:
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            for fp in (Path(directorio) / c).glob(ext):
                img = np.array(Image.open(fp).convert('RGB').resize(size))
                norm = chromatic_normalize(img)
                iso = norm.copy()
                iso[mseg_mask(img, size) == 0] = 0
                originales.append(norm)
                hojas.append(iso)
                etiquetas.append(idx[c])
    return clases, np.stack(originales), np.stack(hojas), np.array(etiquetas)

In [ ]:
def predecir_keras(ruta, Xo, Xl, batch=16):
    modelo = tf.keras.models.load_model(ruta, compile=False)
    salidas = modelo.predict([Xo.astype(np.float32), Xl.astype(np.float32)], batch_size=batch, verbose=0)
    tf.keras.backend.clear_session()
    return salidas


def _cuantizar(x, detalle):
    escala, cero = detalle['quantization']
    if escala == 0:
        return x.astype(detalle['dtype'])
    limites = np.iinfo(detalle['dtype'])
    return np.clip(np.round(x / escala + cero), limites.min, limites.max).astype(detalle['dtype'])


def _descuantizar(y, detalle):
    escala, cero = detalle['quantization']
    if escala == 0:
        return y.astype(np.float32)
    return (y.astype(np.float32) - cero) * escala


def predecir_tflite(ruta, Xo, Xl):
    interp = tf.lite.Interpreter(model_path=str(ruta))
    interp.allocate_tensors()
    entradas = interp.get_input_details()
    salida = interp.get_output_details()[0]
    claves = ('hoja', 'aislada', 'leaf')
    resultados = []
    for i in range(len(Xo)):
        for detalle in entradas:
            fuente = Xl[i] if any(k in detalle['name'].lower() for k in claves) else Xo[i]
            dato = fuente.astype(np.float32)[np.newaxis]
            if detalle['dtype'] in (np.uint8, np.int8):
                dato = _cuantizar(dato, detalle)
            interp.set_tensor(detalle['index'], dato.astype(detalle['dtype']))
        interp.invoke()
        resultados.append(_descuantizar(interp.get_tensor(salida['index'])[0], salida))
    return np.stack(resultados)


def metricas(probs, y, binario):
    pred = (probs.reshape(-1) >= 0.5).astype(int) if binario else probs.argmax(axis=1)
    return accuracy_score(y, pred), f1_score(y, pred, average='macro', zero_division=0)

In [ ]:
import pandas as pd

filas = []
for etiqueta, test_dir, size, binario, claves in [
        ('M1 (estado sanitario)', TEST_BIN, (240, 240), True, ('m1_keras', 'm1_f32', 'm1_int8')),
        ('M2 (patogeno)', TEST_PAT, (224, 224), False, ('m2_keras', 'm2_f32', 'm2_int8'))]:
    clases, Xo, Xl, y = construir_entradas(test_dir, size)
    print(f'{etiqueta}: {len(y)} imagenes | clases {clases}')
    if binario:
        pos = next((i for i, c in enumerate(clases) if 'enferm' in c.lower()), 1)
        y_eval = (y == pos).astype(int)
    else:
        pos = None
        y_eval = y
    referencia = None
    for variante, clave in zip(('Keras', 'TFLite float32', 'TFLite int8'), claves):
        ruta = MODELOS[clave]
        probs = predecir_keras(ruta, Xo, Xl) if clave.endswith('keras') else predecir_tflite(ruta, Xo, Xl)
        if binario and probs.shape[-1] == 1 and pos == 0:
            probs = 1.0 - probs
        acc, f1 = metricas(probs, y_eval, binario)
        if referencia is None:
            referencia = (acc, f1)
        filas.append({'Modelo': etiqueta, 'Variante': variante,
                      'Tamano (MB)': round(Path(ruta).stat().st_size / 1e6, 2),
                      'Exactitud': round(acc, 4), 'F1 macro': round(f1, 4),
                      'D Exactitud': round(acc - referencia[0], 4),
                      'D F1 macro': round(f1 - referencia[1], 4)})
        print(f'  {variante:15s} exactitud={acc:.4f}  F1={f1:.4f}')

tabla = pd.DataFrame(filas)
tabla.to_csv(OUT / 'evaluacion_variantes.csv', index=False)
json.dump(filas, open(OUT / 'evaluacion_variantes.json', 'w'), indent=2, ensure_ascii=False)
tabla

In [ ]:
for modelo in tabla['Modelo'].unique():
    sub = tabla[tabla['Modelo'] == modelo]
    caida = float(sub[sub['Variante'] == 'TFLite int8']['D F1 macro'].iloc[0])
    if abs(caida) < 0.005:
        juicio = 'la cuantizacion no altera de forma apreciable el desempeno'
    elif abs(caida) < 0.02:
        juicio = 'la cuantizacion introduce una perdida menor, aceptable para despliegue movil'
    else:
        juicio = 'la cuantizacion introduce una perdida relevante que conviene reportar'
    print(f'{modelo}: D F1 macro int8 = {caida:+.4f} -> {juicio}.')